## Imports

In [1]:
import torch
import torch.nn as nn
import math
from transformers import LlamaForCausalLM, AutoTokenizer, LlamaConfig, BitsAndBytesConfig
from transformers.models.llama.modeling_llama import LlamaAttention, LlamaMLP, LlamaRMSNorm
from huggingface_hub import login
import ipywidgets as widgets
from IPython.display import display, clear_output
import logging

## Config

In [2]:
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Hugging Face Authentication (Replace with your token)
HF_TOKEN = "Your_HF_Token" # Replace with your actual token

# Model Configuration
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

## Hyperparams

In [3]:
# iRoPE Hyperparameters
CHUNK_SIZE = 2048  # Local attention chunk size
ALPHA = 8192     # α for temperature scaling
BETA = 0.1       # β for temperature scaling
GAMMA = 0.5      # For power-law scaling (if used)
SCALING_TYPE = "log" # Scaling function type ("log", "linear", "exp", "sigmoid", "power")
MAX_SEQ_LEN = 16384 # Maximum sequence length per processing chunk (adjust based on GPU memory)
SIMULATED_CONTEXT_LENGTH = 100_000 # Smaller simulation for faster testing (adjust to 10M if needed)
ROPE_THETA = 500000.0 # Default for LLaMA 3.2 (confirm from specific model config if necessary)

# Device Selection
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

## Rotary Position Embeddings

In [4]:
def apply_rotary_pos_emb(q, k, seq_len, head_dim, rope_theta=ROPE_THETA):
    """Applies Rotary Position Embeddings (RoPE) to query and key tensors."""
    device = q.device
    dtype = q.dtype  # Get the dtype from input tensor
    
    position_ids = torch.arange(seq_len, dtype=torch.long, device=device).unsqueeze(1)
    indices = torch.arange(0, head_dim // 2, dtype=dtype, device=device)  # Match dtype
    freqs = 1.0 / (rope_theta ** (2 * indices / head_dim))
    
    angles = position_ids.float() * freqs  # Cast position_ids to float
    
    cos_angles = torch.cos(angles).to(dtype)  # Convert back to original dtype
    sin_angles = torch.sin(angles).to(dtype)

    # Expand dims for broadcasting: [1, 1, seq_len, head_dim // 2]
    cos_angles = cos_angles.unsqueeze(0).unsqueeze(0)
    sin_angles = sin_angles.unsqueeze(0).unsqueeze(0)

    # Reshape q and k to view pairs of dimensions
    # q: [B, num_heads, L, head_dim] -> [B, num_heads, L, head_dim // 2, 2]
    q_reshaped = q.float().reshape(*q.shape[:-1], -1, 2)
    k_reshaped = k.float().reshape(*k.shape[:-1], -1, 2)

    # Apply rotation
    q_out = torch.zeros_like(q_reshaped)
    k_out = torch.zeros_like(k_reshaped)

    q_out[..., 0] = q_reshaped[..., 0] * cos_angles - q_reshaped[..., 1] * sin_angles
    q_out[..., 1] = q_reshaped[..., 1] * cos_angles + q_reshaped[..., 0] * sin_angles
    k_out[..., 0] = k_reshaped[..., 0] * cos_angles - k_reshaped[..., 1] * sin_angles
    k_out[..., 1] = k_reshaped[..., 1] * cos_angles + k_reshaped[..., 0] * sin_angles

    q_rot = q_out.flatten(3).to(dtype)
    k_rot = k_out.flatten(3).to(dtype)
    return q_rot, k_rot


def create_causal_attention_mask(batch_size, seq_length, device, dtype):
    """Creates a causal attention mask."""
    mask = torch.full((seq_length, seq_length), 
                      dtype=dtype,  # Use passed dtype
                      fill_value=torch.finfo(dtype).min, 
                      device=device)
    mask_cond = torch.arange(mask.size(-1), device=device)
    mask.masked_fill_(mask_cond < (mask_cond + 1).view(mask.size(-1), 1), 0)
    mask = mask.unsqueeze(0).unsqueeze(0)
    mask = mask.expand(batch_size, 1, seq_length, seq_length)
    return mask

## Local Attention With RoPE

In [5]:
class LocalAttentionWithRoPE(LlamaAttention):
    """LLaMA Attention modified for local (chunked) attention with RoPE."""
    def __init__(self, config: LlamaConfig, layer_idx: int, chunk_size: int):
        # Initialize using the parent LlamaAttention constructor FIRST
        super().__init__(config=config, layer_idx=layer_idx)
        self.chunk_size = chunk_size

        # These might be needed by the forward method or even for self.scale below
        self.num_heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.num_key_value_heads = config.num_key_value_heads
        self.hidden_size = config.hidden_size

        # Now self.head_dim is guaranteed to be defined before being used here
        self.scale = self.head_dim ** -0.5

        logging.info(f"Layer {layer_idx}: Initialized LocalAttentionWithRoPE (chunk_size={chunk_size})")


    def forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor | None = None, position_ids: torch.LongTensor | None = None, output_attentions: bool = False, use_cache: bool = False, **kwargs):
        B, L, D = hidden_states.shape
        current_dtype = hidden_states.dtype  # Get current dtype
        
        if use_cache:
            logging.warning("Cache not implemented for LocalAttentionWithRoPE. use_cache=True ignored.")
            use_cache = False
        
        # Process projections
        q = self.q_proj(hidden_states)
        k = self.k_proj(hidden_states)
        v = self.v_proj(hidden_states)
        
        q = q.view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, L, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, L, self.num_key_value_heads, self.head_dim).transpose(1, 2)

        q, k = apply_rotary_pos_emb(q, k, L, self.head_dim)
        
        k = k.repeat_interleave(self.num_heads // self.num_key_value_heads, dim=1)
        v = v.repeat_interleave(self.num_heads // self.num_key_value_heads, dim=1)
        
        # Chunked attention
        attn_outputs = []
        for i in range(0, L, self.chunk_size):
            q_chunk = q[:, :, i:i+self.chunk_size, :]
            k_chunk = k[:, :, i:i+self.chunk_size, :]
            v_chunk = v[:, :, i:i+self.chunk_size, :]

            attn_scores = torch.matmul(q_chunk, k_chunk.transpose(-1, -2)) * self.scale
            if attention_mask is not None:
                attn_scores += attention_mask[:, :, i:i+self.chunk_size, i:i+self.chunk_size]
            attn_probs = torch.softmax(attn_scores, dim=-1)
            attn_out = torch.matmul(attn_probs, v_chunk)  # [B, num_heads, chunk_size, head_dim]
            attn_outputs.append(attn_out)

        attn_out = torch.cat(attn_outputs, dim=2)  # [B, num_heads, L, head_dim]
        attn_out = attn_out.transpose(1, 2).reshape(B, L, -1)
        attn_out = self.o_proj(attn_out)
        
        attn_weights_reshaped = attn_probs if output_attentions else None
        return attn_out, None, attn_weights_reshaped

## Global Attention With Temp Scaling (No RoPE)

In [6]:
class GlobalAttentionWithTempScaling(nn.Module):
    """Global Attention (no RoPE) with Inference-Time Temperature Scaling."""
    def __init__(self, config: LlamaConfig, layer_idx: int, alpha: float, beta: float, gamma: float, scaling_type: str):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.num_heads = config.num_attention_heads
        self.num_key_value_heads = config.num_key_value_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.hidden_size = config.hidden_size
        self.scale = self.head_dim ** -0.5
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.scaling_type = scaling_type

        if self.num_heads % self.num_key_value_heads != 0:
             raise ValueError(f"num_heads ({self.num_heads}) must be divisible by num_key_value_heads ({self.num_key_value_heads})")

        kv_dim = self.num_key_value_heads * self.head_dim

        # Use LLaMA's projection layers
        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=config.attention_bias)
        self.k_proj = nn.Linear(self.hidden_size, kv_dim, bias=config.attention_bias)
        self.v_proj = nn.Linear(self.hidden_size, kv_dim, bias=config.attention_bias)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=config.attention_bias)
        logging.info(f"Layer {layer_idx}: Initialized GlobalAttentionWithTempScaling (alpha={alpha}, beta={beta}, type={scaling_type})")


    def forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor | None = None, position_ids: torch.LongTensor | None = None, output_attentions: bool = False, use_cache: bool = False, **kwargs):
        B, L, D = hidden_states.shape
        current_dtype = hidden_states.dtype  # Get the current dtype
        
        # Compute Q, K, V
        q = self.q_proj(hidden_states)
        k = self.k_proj(hidden_states)
        v = self.v_proj(hidden_states)
        
        # Reshape for multi-head attention
        q = q.view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, L, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, L, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        
        # Expand k and v for GQA/MQA
        k = k.repeat_interleave(self.num_heads // self.num_key_value_heads, dim=1)
        v = v.repeat_interleave(self.num_heads // self.num_key_value_heads, dim=1)
        
        positions = torch.arange(L, device=hidden_states.device, dtype=current_dtype)  # Match dtype
        scaling_factor = torch.ones_like(positions)
        
        if self.scaling_type == "log":
            scaling_factor = 1 + torch.log(torch.floor(positions / self.alpha) + 1) * self.beta
        elif self.scaling_type == "linear":
            scaling_factor = 1 + (positions / self.alpha) * self.beta
        elif self.scaling_type == "exp":
            scaling_factor = 1 + torch.exp((positions / self.alpha) - 1) * self.beta
        elif self.scaling_type == "sigmoid":
            scaling_factor = 1 + torch.sigmoid((positions / self.alpha) - (L / (2*self.alpha))) * self.beta
        elif self.scaling_type == "power":
            scaling_factor = 1 + (positions / self.alpha).pow(self.gamma) * self.beta
            
        # Apply scaling factor with correct dtype
        scaling_factor = scaling_factor.to(current_dtype)
        q = q * scaling_factor.view(1, 1, L, 1)
        
        attn_scores = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        
        if attention_mask is not None:
            attn_scores = attn_scores + attention_mask.to(current_dtype)
        
        # Softmax in float32 for stability, then back to original dtype
        attn_probs = torch.softmax(attn_scores, dim=-1, dtype=torch.float32).to(current_dtype)
        
        attn_output = torch.matmul(attn_probs, v)
        
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, L, self.hidden_size)
        attn_output = self.o_proj(attn_output)
        
        attn_weights_reshaped = attn_probs if output_attentions else None
        
        return attn_output, None, attn_weights_reshaped

## Llama Layer With IRoPE

In [7]:
class LlamaLayerWithIRoPE(nn.Module):
    """A LLaMA Decoder Layer modified to use interleaved Local and Global Attention."""
    def __init__(self, config: LlamaConfig, layer_idx: int, chunk_size: int, alpha: float, beta: float, gamma: float, scaling_type: str):
        super().__init__()
        self.layer_idx = layer_idx
        self.hidden_size = config.hidden_size
        self.use_local = layer_idx % 2 == 0 # Interleave: Even layers -> Local, Odd layers -> Global

        if self.use_local:
            self.self_attn = LocalAttentionWithRoPE(config=config, layer_idx=layer_idx, chunk_size=chunk_size)
        else:
            self.self_attn = GlobalAttentionWithTempScaling(config=config, layer_idx=layer_idx, alpha=alpha, beta=beta, gamma=gamma, scaling_type=scaling_type)

        self.mlp = LlamaMLP(config)

        self.input_layernorm = LlamaRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = LlamaRMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.LongTensor | None = None,
        output_attentions: bool | None = False,
        use_cache: bool | None = False,
        **kwargs, # Accept potential extra arguments
    ) :
        residual = hidden_states
        hidden_states_norm = self.input_layernorm(hidden_states)

        # Self Attention
        attn_output, _, attn_weights = self.self_attn(
            hidden_states=hidden_states_norm,
            attention_mask=attention_mask,
            position_ids=position_ids,
            output_attentions=output_attentions,
            use_cache=use_cache,
            **kwargs
        )

        # Residual connection + Attention output
        hidden_states = residual + attn_output

        # MLP block
        residual = hidden_states
        hidden_states_norm = self.post_attention_layernorm(hidden_states)
        hidden_states_mlp = self.mlp(hidden_states_norm)

        # Residual connection + MLP output
        hidden_states = residual + hidden_states_mlp

        outputs = (hidden_states,)
        if output_attentions:
            outputs += (attn_weights,)

        return outputs # Return tuple consistent with HF Layer output

## Llama With IRoPE

In [8]:
class LlamaWithIRoPE(LlamaForCausalLM):
    """LLaMA Model using interleaved iRoPE layers."""
    def __init__(self, config: LlamaConfig, chunk_size: int, alpha: float, beta: float, gamma: float, scaling_type: str):
        super().__init__(config) # Initialize the base LlamaForCausalLM

        # Replace standard layers with iRoPE layers
        self.model.layers = nn.ModuleList([
            LlamaLayerWithIRoPE(config, layer_idx, chunk_size, alpha, beta, gamma, scaling_type)
            for layer_idx in range(config.num_hidden_layers)
        ])

        logging.info(f"Initialized LlamaWithIRoPE with {config.num_hidden_layers} interleaved layers.")
        # Need to call post_init if necessary (usually handles weight tying)
        self.post_init()

## IRoPE Llama Interface

In [9]:
class IRoPELlamaInterface:
    def __init__(self, model_name, hf_token, chunk_size, alpha, beta, gamma, scaling_type, max_seq_len, device, dtype):
        self.model_name = model_name
        self.hf_token = hf_token
        self.chunk_size = chunk_size
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.scaling_type = scaling_type
        self.max_seq_len = max_seq_len # Max length for *processing chunks*
        self.device = device
        self.dtype = dtype

        self.model = None
        self.tokenizer = None

        self._login_hf()
        self._load_model_and_tokenizer()
        self._setup_ui()

    def _login_hf(self):
        try:
            login(self.hf_token)
            logging.info("Successfully logged into Hugging Face Hub.")
        except Exception as e:
            logging.error(f"Hugging Face login failed: {e}")

    def _load_model_and_tokenizer(self):
        logging.info(f"Loading tokenizer for {self.model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        if self.tokenizer.pad_token is None:
            logging.warning("Tokenizer does not have a pad token. Setting to EOS token.")
            self.tokenizer.pad_token = self.tokenizer.eos_token

        logging.info(f"Loading original model {self.model_name}...")
        try:
            original_model = LlamaForCausalLM.from_pretrained(
                self.model_name,
                torch_dtype=self.dtype, # Load in specified dtype
                device_map=self.device # Let HF handle device placement initially
            )
            original_model.eval() # Set to evaluation mode
            self.config = original_model.config # Get config from loaded model

            logging.info("Original model loaded. Building iRoPE model...")
            self.model = LlamaWithIRoPE(
                self.config,
                chunk_size=self.chunk_size,
                alpha=self.alpha,
                beta=self.beta,
                gamma=self.gamma,
                scaling_type=self.scaling_type
            )

            self._transfer_weights(original_model)

            # Move the final iRoPE model to the desired device and dtype
            # (HF device_map might have spread the original model)
            self.model = self.model.to(device=self.device, dtype=self.dtype)
            self.model.eval() # Set iRoPE model to eval mode

            logging.info(f"iRoPE model created, weights transferred, and moved to {self.device}.")
            del original_model # Free memory
            torch.cuda.empty_cache() # Clear cache if using GPU

        except Exception as e:
            logging.error(f"Error loading model or transferring weights: {e}")
            raise

    def _transfer_weights(self, original_model):
        logging.info("Starting weight transfer...")
        # Transfer Embeddings
        self.model.model.embed_tokens.load_state_dict(original_model.model.embed_tokens.state_dict())

        # Transfer Transformer Layers
        for layer_idx, irope_layer in enumerate(self.model.model.layers):
            original_layer = original_model.model.layers[layer_idx]

            # Copy LayerNorm weights
            irope_layer.input_layernorm.load_state_dict(original_layer.input_layernorm.state_dict())
            irope_layer.post_attention_layernorm.load_state_dict(original_layer.post_attention_layernorm.state_dict())

            # Copy Attention weights (Q, K, V, O projections)
            # These weights exist in both LocalAttentionWithRoPE and GlobalAttentionWithTempScaling
            irope_layer.self_attn.q_proj.load_state_dict(original_layer.self_attn.q_proj.state_dict())
            irope_layer.self_attn.k_proj.load_state_dict(original_layer.self_attn.k_proj.state_dict())
            irope_layer.self_attn.v_proj.load_state_dict(original_layer.self_attn.v_proj.state_dict())
            irope_layer.self_attn.o_proj.load_state_dict(original_layer.self_attn.o_proj.state_dict())

            # Copy MLP weights correctly
            irope_layer.mlp.gate_proj.load_state_dict(original_layer.mlp.gate_proj.state_dict())
            irope_layer.mlp.up_proj.load_state_dict(original_layer.mlp.up_proj.state_dict())
            irope_layer.mlp.down_proj.load_state_dict(original_layer.mlp.down_proj.state_dict())

            if (layer_idx + 1) % 5 == 0: # Log progress every few layers
                 logging.info(f"Transferred weights for layer {layer_idx+1}/{len(self.model.model.layers)}")


        # Transfer Final LayerNorm
        self.model.model.norm.load_state_dict(original_model.model.norm.state_dict())

        # Transfer LM Head
        self.model.lm_head.load_state_dict(original_model.lm_head.state_dict())
        logging.info("Weight transfer completed.")


    def process_long_context(self, input_text: str, simulated_context_length: int):
        """ Tokenizes, simulates long context, processes in chunks, and generates output. """
        if not self.model or not self.tokenizer:
            logging.error("Model or Tokenizer not loaded.")
            return "Error: Model not ready."

        logging.info(f"Processing input (simulating {simulated_context_length} tokens)...")
        # Tokenize the input
        inputs = self.tokenizer(input_text, return_tensors="pt", truncation=False)
        input_ids_single = inputs["input_ids"].to(self.device)

        if input_ids_single.shape[1] == 0:
             logging.warning("Input text tokenized to an empty sequence.")
             return "Input is empty after tokenization."

        # Simulate a long context by repeating the input (adjust this logic if needed)
        num_repeats = max(1, simulated_context_length // input_ids_single.shape[1])
        input_ids = input_ids_single.repeat(1, num_repeats)
        total_length = input_ids.shape[1]
        logging.info(f"Original tokens: {input_ids_single.shape[1]}, Repeated tokens: {total_length}")

        # Process in chunks to handle memory constraints
        all_logits = []
        last_hidden_state = None

        for start in range(0, total_length, self.max_seq_len):
            end = min(start + self.max_seq_len, total_length)
            chunk_input_ids = input_ids[:, start:end]
            chunk_seq_len = chunk_input_ids.shape[1]
            batch_size = chunk_input_ids.shape[0]
            logging.info(f"Processing chunk: start={start}, end={end}, len={chunk_seq_len}")

            # Create a causal attention mask for the chunk
            causal_mask = create_causal_attention_mask(batch_size, chunk_seq_len, self.device, self.dtype)
            chunk_attention_mask = causal_mask # Use the causal mask directly

            with torch.no_grad():
                 # Pass position_ids explicitly if needed, otherwise model calculates them
                 # position_ids_chunk = torch.arange(start, end, dtype=torch.long, device=self.device).unsqueeze(0)
                 outputs = self.model(
                     input_ids=chunk_input_ids,
                     attention_mask=chunk_attention_mask, # Pass the combined mask
                     output_hidden_states=True, # Request hidden states
                     use_cache=False # Cache is disabled in custom layers
                 )
                 # Store logits or hidden states from the chunk
                 # Logits are typically needed for generation
                 chunk_logits = outputs.logits # Shape [B, chunk_seq_len, vocab_size]
                 all_logits.append(chunk_logits)
                 last_hidden_state = outputs.hidden_states[-1] # Keep track of last hidden state if needed

                 # Memory management
                 del outputs
                 if self.device == 'cuda': torch.cuda.empty_cache()


        logging.info("Finished processing chunks.")
        # Concatenate logits from all chunks along the sequence length dimension
        full_logits = torch.cat(all_logits, dim=1) # [B, total_length, vocab_size]
        # Use the logits of the *very last* token to predict the next token
        predicted_token_id = torch.argmax(full_logits[:, -1, :], dim=-1) # [B]
        output_ids = torch.cat([input_ids_single[:,], predicted_token_id.unsqueeze(1)], dim=1)

        # Decode the output sequence
        logging.info("Decoding generated sequence...")
        output_text = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)

        return output_text.replace(input_text,"") # Return only the newly generated part


    def _setup_ui(self):
        """Creates the ipywidgets UI."""
        self.input_box = widgets.Textarea(
            value="Tell me a story about a futuristic city.",
            placeholder="Type your input here...",
            description="Input:",
            layout={'width': '90%', 'height': '100px'}
        )
        self.context_slider = widgets.IntSlider(
            value=SIMULATED_CONTEXT_LENGTH,
            min=5,
            max=10_000_000, # Allow up to 10M
            step=1000,
            description='Simulated Context:',
            style={'description_width': 'initial'},
            layout={'width': '90%'}
        )

        self.output_box = widgets.Output(layout={'border': '1px solid black', 'padding': '5px', 'width': '90%', 'height': '300px', 'overflow_y': 'auto'})
        self.button = widgets.Button(description="Generate", button_style="primary", layout={'width': '150px'})
        self.status_label = widgets.Label(value="Status: Ready")

        self.button.on_click(self._on_button_clicked)

    def _on_button_clicked(self, b):
        """Handles button click event."""
        input_text = self.input_box.value
        sim_context = self.context_slider.value
        self.status_label.value = f"Status: Processing (Simulating {sim_context} tokens)..."
        self.button.disabled = True
        with self.output_box:
            clear_output()
            print(f"Input:\n{input_text}\n" + "-"*20)
            print("Processing...")
            try:
                 output_text = self.process_long_context(input_text, sim_context)
                 clear_output() # Clear "Processing..." message
                 print(f"Input:\n{input_text}\n" + "-"*20)
                 print(f"Generated Output (Simulated Context: {sim_context}):")
                 print(output_text)
                 self.status_label.value = "Status: Done"
            except Exception as e:
                 clear_output() # Clear "Processing..." message
                 print(f"Input:\n{input_text}\n" + "-"*20)
                 print(f"An error occurred: {e}")
                 logging.error(f"Error during generation: {e}", exc_info=True)
                 self.status_label.value = "Status: Error"
            finally:
                 self.button.disabled = False


    def display_ui(self):
        """Displays the created UI."""
        display(self.input_box)
        display(self.context_slider)
        display(self.button)
        display(self.status_label)
        display(self.output_box)

if __name__ == "__main__":
    try:
        get_ipython()
        is_ipython = True
        logging.info("IPython environment detected. Proceeding with UI.")
    except NameError:
        is_ipython = False
        logging.warning("Not running in an IPython environment. UI will not be displayed.")
        print("This script includes an interactive UI that requires an IPython/Jupyter environment.")
        print("To run with the UI, please execute this script in Jupyter Notebook, JupyterLab, Google Colab, or similar.")

    if is_ipython:
        interface = IRoPELlamaInterface(
            model_name=MODEL_NAME,
            hf_token=HF_TOKEN,
            chunk_size=CHUNK_SIZE,
            alpha=ALPHA,
            beta=BETA,
            gamma=GAMMA,
            scaling_type=SCALING_TYPE,
            max_seq_len=MAX_SEQ_LEN,
            device=DEVICE,
            dtype=TORCH_DTYPE
        )

        interface.display_ui()

2025-04-14 20:06:55,215 - INFO - IPython environment detected. Proceeding with UI.
2025-04-14 20:06:55,268 - INFO - Successfully logged into Hugging Face Hub.
2025-04-14 20:06:55,270 - INFO - Loading tokenizer for meta-llama/Llama-3.2-3B-Instruct...
2025-04-14 20:06:55,755 - WARNING - Tokenizer does not have a pad token. Setting to EOS token.
2025-04-14 20:06:55,756 - INFO - Loading original model meta-llama/Llama-3.2-3B-Instruct...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2025-04-14 20:07:03,733 - INFO - Original model loaded. Building iRoPE model...
2025-04-14 20:07:41,411 - INFO - Layer 0: Initialized LocalAttentionWithRoPE (chunk_size=2048)
2025-04-14 20:07:41,938 - INFO - Layer 1: Initialized GlobalAttentionWithTempScaling (alpha=8192, beta=0.1, type=log)
2025-04-14 20:07:42,466 - INFO - Layer 2: Initialized LocalAttentionWithRoPE (chunk_size=2048)
2025-04-14 20:07:42,998 - INFO - Layer 3: Initialized GlobalAttentionWithTempScaling (alpha=8192, beta=0.1, type=log)
2025-04-14 20:07:43,524 - INFO - Layer 4: Initialized LocalAttentionWithRoPE (chunk_size=2048)
2025-04-14 20:07:44,052 - INFO - Layer 5: Initialized GlobalAttentionWithTempScaling (alpha=8192, beta=0.1, type=log)
2025-04-14 20:07:44,583 - INFO - Layer 6: Initialized LocalAttentionWithRoPE (chunk_size=2048)
2025-04-14 20:07:45,109 - INFO - Layer 7: Initialized GlobalAttentionWithTempScaling (alpha=8192, beta=0.1, type=log)
2025-04-14 20:07:45,638 - INFO - Layer 8: Initialized LocalAttention

Textarea(value='Tell me a story about a futuristic city.', description='Input:', layout=Layout(height='100px',…

IntSlider(value=100000, description='Simulated Context:', layout=Layout(width='90%'), max=10000000, min=5, ste…

Button(button_style='primary', description='Generate', layout=Layout(width='150px'), style=ButtonStyle())

Label(value='Status: Ready')

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…